In [4]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(bnlearn)')

In [5]:
#set up for experimentation
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_HC"
output_dir.mkdir(parents=True,exist_ok=True)

In [6]:
import time
#run hc function
def run_hc(df, seed=1):
    #remove NA rows
    df_clean = df.dropna().copy()

    start = time.time()
    with (ro.default_converter+pandas2ri.converter).context():
        ro.globalenv["df"]=conversion.py2rpy(df_clean)
        ro.globalenv["seed"]= seed

        ro.r('''
        library(bnlearn)
        data_df <- as.data.frame(df)
        
        any_disc <- FALSE
        any_cont <- FALSE

        for (i in seq_along(data_df)) {
          col <- data_df[[i]]
          if (is.numeric(col)) {
            vals <- unique(col[!is.na(col)])
            if (length(vals) <= 6 && all(abs(vals - round(vals)) < 1e-8)) {
              data_df[[i]] <- factor(col)
              any_disc <- TRUE
            } else {
              data_df[[i]] <- as.numeric(col)
              any_cont <- TRUE
            }
          } else {
            data_df[[i]] <- factor(col)
            any_disc <- TRUE
          }
        }

        #detect if there are mixed types, continuous only, or discrete only
        if (any_disc && any_cont) {
          chosen_score <- "bic-cg"    # mixed conditional Gaussian [web:148][web:154]
        } else if (any_disc && !any_cont) {
          chosen_score <- "bde"       # discrete-only
        } else if (!any_disc && any_cont) {
          chosen_score <- "bic-g"     # Gaussian-only
        } else {
          stop("No usable variables (neither discrete nor continuous).")
        }
        
        set.seed(seed)

        #run hill climbing with conditional score
        dag_hc <- hc(data_df, score=chosen_score)

        adj <- amat(dag_hc)
        nodes <- colnames(adj)
        ''')
        end = time.time()
        
        #convert adjacency matrix in R back to python
        adj = conversion.rpy2py(ro.r("adj"))
        nodes=list(ro.r("nodes"))

    #coerce adjacency matrix into numeric 2D array
    #print(f"HC took {(end - start)/60:.2f} minutes")
    adj=np.asarray(adj, dtype=int)
    return adj,nodes

In [4]:
import networkx as nx
import matplotlib.pyplot as plt

#graph drawing functions
def draw_graph(adj, nodes, out_path):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i,j]==1:
                G.add_edge(src,tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [5]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [6]:
csv_path = output_dir/ "NIJ_graph_HC_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        adj, _ = run_hc(df_subset, seed=1)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

score = incompatibility_score(A, k=5, n_subsets=50, seed=42)
print("Approx. incompatibility score:", score)


HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 

In [7]:
incompat_score = 4.9

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))

goodness(incompat_score, 5)

% of edges disagreeing on avg: 24.500000000000004


In [8]:
score1 = incompatibility_score(A, k=10, n_subsets=50,seed=42)
goodness(score1, 10)

HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 minutes
HC took 0.01 

In [9]:
score2 = incompatibility_score(A, k=5, n_subsets=100,seed=42)
goodness(score2, 5)

HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 minutes
HC took 0.00 

In [8]:
csv_path = output_dir/ "NIJ_graph_HC_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        adj, _ = run_hc(df_subset, seed=1)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))
    return(frac*100)

In [16]:
csv_path = output_dir/ "NIJ_graph_HC_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

seeds = [42,7,12]
incompat_scores = []
disagree = []
for seed in seeds:
    score = incompatibility_score(A, k = 5, n_subsets=50, seed=seed)
    incompat_scores.append(score)
    disagree.append(goodness(score, 5))

% of edges disagreeing on avg: 24.500000000000004
% of edges disagreeing on avg: 26.3
% of edges disagreeing on avg: 28.4


In [17]:
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 0.3187475490101843
standard dev of disagreement percentage: 1.5937377450509207


In [18]:
incompat_scores

[4.9, 5.26, 5.68]

In [19]:
csv_path = nij_root / "NIJ_lean_compact_onehot.csv"
df_full = pd.read_csv(csv_path)
node_labels = list(df_full.columns)
d = len(node_labels)

def bootstrap_edge_stability(df, B=20, seed=0):
    rng = np.random.default_rng(seed)
    edge_counts = np.zeros((d, d), dtype=int)

    for b in range(B):
        #sample B rows with replacement from original learning data
        idx = rng.integers(low=0, high=len(df), size=len(df))
        df_boot = df.iloc[idx, :]

        #run DAGSLAM on each bootstrap sample
        W_est, _ = run_hc(df_boot, seed=1)           # shape (d, d), aligned with columns
        W_bin = (np.asarray(W_est) != 0).astype(int)

        #accumulate edge counts
        edge_counts += W_bin

    #convert to percentage appearances
    edge_freq = edge_counts / B
    return edge_freq
    
edge_freq_hc = bootstrap_edge_stability(df_full, B=20, seed=42)

#save result to CSV
edge_freq_df = pd.DataFrame(edge_freq_hc, index=node_labels, columns=node_labels)
edge_freq_df.to_csv(output_dir / "NIJ_HC_edge_stability.csv")
print("Number of edges with freq >= 0.5:",
      np.sum(edge_freq_hc >= 0.5))
print("Number of edges with freq >= 0.8:",
      np.sum(edge_freq_hc >= 0.8))


/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '


Number of edges with freq >= 0.5: 75
Number of edges with freq >= 0.8: 56


In [9]:
csv_path = output_dir/ "NIJ_graph_HC_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

subset_sizes=[5,10,15]
incompat_scores = []
disagree = []
for size in subset_sizes:
    score = incompatibility_score(A, k = size, n_subsets=50, seed=42)
    incompat_scores.append(score)
    disagree.append(goodness(score, size))

% of edges disagreeing on avg: 24.500000000000004
% of edges disagreeing on avg: 18.933333333333334
% of edges disagreeing on avg: 13.438095238095238


In [10]:
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 9.523038730713356
standard dev of disagreement percentage: 4.516035090683777


In [11]:
incompat_scores

[4.9, 17.04, 28.22]